In [ ]:
import os
import pandas as pd
import numpy as np
import json
from collections import Counter
from tqdm import tqdm

# Load original dataset

In [ ]:
df = pd.read_csv("hf://datasets/wwbrannon/twinviews-13k/twinviews-13k.csv")

right_stances = df["r"].to_list()
left_stances = df["l"].to_list()

# Minimal Preprocessing

In [ ]:
def add_point(s):
    s=s.strip()
    if s.endswith("."):
        return s
    else:
        return s+"."

df["l"] = df["l"].map(add_point)
df["r"] = df["r"].map(add_point)

# Labels merging

In [ ]:
with open("topic_to_bertopic.json", "r") as f:
    topic_to_bertopic = json.load(f)

In [ ]:
df["issue"] = df["topic"].map(topic_to_bertopic)

In [ ]:
df

In [ ]:
bertopics_list = list(df["issue"])
mc_topics = Counter(bertopics_list).most_common()

labels = [mc[0] for mc in mc_topics]
values = [mc[1] for mc in mc_topics]

n_topics = 10
selected_topics = labels[:n_topics]

selected_topics

In [ ]:
df_filtered = df[[(issue in selected_topics) for issue in df["issue"]]]
df_filtered["statement_id"] = np.arange(len(df_filtered))
df_filtered

# Flattening left-/right- stances dataframe

In [ ]:
dfs = []
for lab in ["l","r"]:
    l_df = df_filtered[[lab, "topic", "issue", "statement_id"]].copy()
    l_df.rename({lab:'statement'}, inplace=True, axis=1)
    l_df["label"] = lab
    dfs += [l_df]


df_flattened = pd.concat(dfs)

In [ ]:
df_flattened

In [ ]:
df_flattened["data_id"] = np.arange(len(df_flattened))

In [ ]:
df_flattened.to_csv(
    "../preprocessed/twinviews_prepro.csv",
    index=False
)